# SIR Model Experiment: Discovering R₀ = β/γ

This notebook demonstrates the Degeneracy Distillery on the classic SIR epidemiological model.

**Ground truth:** From observed infection curves, the basic reproduction number R₀ = β/γ 
is the only identifiable parameter combination. The individual rates β and γ cannot be 
separately identified from incidence data alone.

**Goal:** Automatically discover the symbolic expression R₀ = β/γ using our pipeline.

## Pipeline Overview
1. Define SIR simulator (scipy ODE solver)
2. Generate training/test data (10K parameter-data pairs)
3. Train Fisher Network ensemble
4. Train flattening normalizing flow
5. Run symbolic regression
6. Postprocess and compare with ground truth

In [ ]:
# Colab Setup: Clone and install packages
!git clone https://github.com/tlmakinen/degeneracy_distillery.git
%cd /content/degeneracy_distillery
!pip install -q -e .
%cd /content/
!git clone https://github.com/DeaglanBartlett/ESR.git
%cd /content/ESR
!pip install -q -e .

# ⚠️ RESTART YOUR SESSION after installation

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
%cd /content/
import esr.generation.generator
print("✓ ESR installed successfully!")
from degeneracy_distillery.sr_utils import fit_and_analyze_sr
print("✓ All imports successful!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jr
from scipy.integrate import odeint
from tqdm import tqdm
import degeneracy_distillery
from degeneracy_distillery.training_loop_fishnets import train_fishnets

plt.rcParams.update({
    'font.size': 14, 'axes.labelsize': 16, 'axes.titlesize': 16,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12,
    'figure.figsize': (8, 6), 'figure.dpi': 150,
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

## 1. Problem Definition & Data Generation

### The SIR Model
The SIR model describes disease spread through a population:
- dS/dt = -β·S·I/N
- dI/dt = β·S·I/N - γ·I  
- dR/dt = γ·I

where β is the transmission rate, γ is the recovery rate, and N is the population size.

### Known Degeneracy
From observed I(t) time series, only R₀ = β/γ determines the epidemic dynamics.
Scaling β → c·β and γ → c·γ simultaneously leaves the dynamics unchanged.

### Parameters
- θ = (β, γ) with β ∈ [0.1, 1.0], γ ∈ [0.05, 0.5]
- Data: Noisy I(t) at 30 time points, normalized by N

In [ ]:
# SIR Model Configuration
key = jr.PRNGKey(42)

# Population and time parameters
N_pop = 1000
I0 = 10
S0 = N_pop - I0
R0_init = 0
n_timepoints = 30
t_max = 50.0
t_obs = np.linspace(0, t_max, n_timepoints)
noise_std = 0.01

# Parameter ranges
BETA_MIN, BETA_MAX = 0.1, 1.0
GAMMA_MIN, GAMMA_MAX = 0.05, 0.5

# Number of simulations
nsims = 10000

def sir_odes(y, t, beta, gamma, N):
    """SIR ODEs."""
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]

def simulate_sir(beta, gamma, noise_key=None):
    """Run SIR simulation and return noisy normalized I(t)."""
    y0 = [S0, I0, R0_init]
    solution = odeint(sir_odes, y0, t_obs, args=(beta, gamma, N_pop))
    I_t = solution[:, 1] / N_pop  # normalized infected fraction
    
    if noise_key is not None:
        noise = jr.normal(noise_key, shape=I_t.shape) * noise_std
        I_t = I_t + np.array(noise)
    
    return I_t

# Generate training data
print(f"Generating {nsims} training simulations...")
key, subkey = jr.split(key)
beta_train = np.random.uniform(BETA_MIN, BETA_MAX, nsims)
gamma_train = np.random.uniform(GAMMA_MIN, GAMMA_MAX, nsims)
theta_train = np.stack([beta_train, gamma_train], axis=1)

keys_train = jr.split(subkey, nsims)
data_train = np.array([
    simulate_sir(theta_train[i, 0], theta_train[i, 1], keys_train[i])
    for i in tqdm(range(nsims), desc="Training sims")
])

# Generate test data
key, subkey = jr.split(key)
beta_test = np.random.uniform(BETA_MIN, BETA_MAX, nsims)
gamma_test = np.random.uniform(GAMMA_MIN, GAMMA_MAX, nsims)
theta_test = np.stack([beta_test, gamma_test], axis=1)

keys_test = jr.split(subkey, nsims)
data_test = np.array([
    simulate_sir(theta_test[i, 0], theta_test[i, 1], keys_test[i])
    for i in tqdm(range(nsims), desc="Test sims")
])

# Convert to JAX arrays for training
theta_train = jnp.array(theta_train)
data_train = jnp.array(data_train)
theta_test = jnp.array(theta_test)
data_test = jnp.array(data_test)

print(f"\nTraining data shape: {data_train.shape}")
print(f"Training theta shape: {theta_train.shape}")
print(f"Test data shape: {data_test.shape}")
print(f"R₀ range: [{(BETA_MIN/GAMMA_MAX):.2f}, {(BETA_MAX/GAMMA_MIN):.2f}]")

In [ ]:
# Plot example SIR curves for different R_0 values
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Example curves
for idx in np.random.choice(nsims, 5, replace=False):
    R0_val = theta_train[idx, 0] / theta_train[idx, 1]
    axes[0].plot(t_obs, data_train[idx], alpha=0.7, label=f'R₀={R0_val:.2f}')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('I(t)/N')
axes[0].set_title('Example infection curves')
axes[0].legend(fontsize=8)

# Parameter space colored by R_0
R0_vals = np.array(theta_train[:, 0] / theta_train[:, 1])
sc = axes[1].scatter(theta_train[:, 0], theta_train[:, 1], c=R0_vals, 
                     s=1, alpha=0.5, cmap='viridis')
plt.colorbar(sc, ax=axes[1], label='R₀ = β/γ')
axes[1].set_xlabel('β')
axes[1].set_ylabel('γ')
axes[1].set_title('Parameter space')

# R_0 distribution
axes[2].hist(R0_vals, bins=50, density=True, alpha=0.7)
axes[2].set_xlabel('R₀ = β/γ')
axes[2].set_ylabel('Density')
axes[2].set_title('R₀ distribution')

plt.tight_layout()
plt.savefig('sir_data_overview.pdf')
plt.show()

## 2. Fisher Network (Fishnet) Training

Training an ensemble of 20 Fisher Networks to predict MLE-like parameters 
and Fisher information matrices from the observed I(t) time series.

In [ ]:
# Train Fishnet ensemble
train_fishnets(
    theta_train, data_train,
    theta_test, data_test,
    num_models=20,
    train_epochs=5000,
    patience=30,
    lr=5e-5,
    train_batch_size=200,
    outdir="fishnets-log-sir"
)

## 3. Normalizing Flow Flattening

Training the normalizing flow to align Fisher geometry.

In [ ]:
from degeneracy_distillery.training_loop_flatten import *

# Load fishnet ensemble (train_fishnets saves fishnets_outputs.npz with keys theta, Fs, ensemble_weights)
data_npz = np.load("fishnets-log-sir/fishnets_outputs.npz")
thetas = jnp.array(data_npz["theta"])
ensemble_weights = data_npz["ensemble_weights"]
F_network_ensemble = jnp.array(data_npz["Fs"])

print("thetas", thetas.shape, "F_network_ensemble", F_network_ensemble.shape)

w, ensemble_w, outputs_flatten = fit_flattening(
    F_network_ensemble,
    thetas,
    ensemble_weights=ensemble_weights,
    hidden_size=256,
    n_layers=10,
    batch_size=250,
    epochs_phase1=10000,
    epochs_phase2=250,
    finetune_epochs=250,
    min_epochs=1200,
    patience=50,
    lr_phase1=2e-6,
    lr_schedule_initial=7e-5,
    lr_decay=0.3,
    lr_finetune=4e-6,
    norm_factor=None,
    norm_method="median_det",
    noise=1e-5,
    seed=0,
    output_prefix="sir_experiment",
    SCALE_THETA=False,
    use_whitening=True,
    do_plot=False,
)

## 4. Preprocessing & Coordinate Alignment

Loading processed data and aligning ensemble predictions.

In [ ]:
from degeneracy_distillery.preprocessing_utils import load_and_process_data

data = load_and_process_data(
    datapath="./",
    filename="sir_experiment.npz",
    num_samps=4000,
    seed=44,
    process_ensemble=True,
    n_d=1.0,
    apply_varimax=True,
    jacobian_sparsity="norm",
    verbose=True,
)

X = data['X']
y = data['y']
y_std = data['y_std']
dy_sr = data['dy_sr']
Fs = data['Fs']
n_params = X.shape[1]

print(f"\nProcessed data shapes:")
print(f"  X (parameters): {X.shape}")
print(f"  y (network outputs): {y.shape}")
print(f"  Fs (Fisher matrices): {Fs.shape}")

### Flattened coordinates: $X$ vs $y$

After `load_and_process_data`, each row is a sample; $X$ holds (masked) parameters in the flattened frame and $y$ the corresponding network outputs. This grid plots $X_i$ vs $y_j$ with $\pm 1\sigma$ error bars on $y$ (subsampled for speed) to see which coordinate pairs are strongly coupled before symbolic regression.

In [ ]:
def plot_flattened_X_vs_y(X, y, y_std, skip=10, save_path=None):
    """Grid of X_i vs y_j with ±1σ error bars on y (subsampled)."""
    X = np.asarray(X)
    y = np.asarray(y)
    y_std = np.asarray(y_std)
    nx, ny = X.shape[1], y.shape[1]
    fig, axs = plt.subplots(nx, ny, figsize=(2.3 * ny, 2.0 * nx), squeeze=False)
    sub = slice(None, None, skip)
    for i in range(nx):
        for j in range(ny):
            ax = axs[i, j]
            ax.errorbar(
                X[sub, i],
                y[sub, j],
                yerr=y_std[sub, j],
                fmt="none",
                ecolor="0.72",
                elinewidth=0.6,
                capsize=0,
                zorder=1,
            )
            ax.scatter(
                X[sub, i],
                y[sub, j],
                s=8,
                c="C0",
                alpha=0.35,
                linewidths=0,
                zorder=2,
            )
            ax.set_xlabel(rf"$X_{{{i}}}$")
            ax.set_ylabel(rf"$y_{{{j}}}$")
            ax.tick_params(labelsize=9)
    fig.suptitle(
        r"Flattened frame: parameters $X$ vs network outputs $y$",
        fontsize=12,
        y=1.01,
    )
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()


plot_flattened_X_vs_y(X, y, y_std, skip=10, save_path="X_vs_y_grid_sir.pdf")

## 5. Symbolic Regression

Running PyOperon symbolic regression to discover identifiable parameter 
combinations. We expect to find expressions involving β/γ (ratio).

Key settings:
- `allowed_symbols` includes `inv` for discovering ratios
- `max_complexity_thresh=14` to keep expressions interpretable

In [ ]:
from degeneracy_distillery.sr_utils import fit_and_analyze_sr

mdl_coords, frob_coords, analysis, split_data = fit_and_analyze_sr(
    X, y, y_std, dy_sr, Fs,
    n_params=n_params,
    parent_dir='./sr_results_sir/',
    test_size=0.5,
    random_state=42,
    shuffle=True,
    time_limit=120,
    max_length=25,
    max_depth=10,
    allowed_symbols='add,mul,pow,constant,variable,exp,logabs,sqrt,inv',
    max_complexity_thresh=14,
    equation_set='pareto',
)

print("\n" + "="*60)
print("SYMBOLIC REGRESSION RESULTS")
print("="*60)
print(f"\nBest MDL coordinates: {mdl_coords}")
print(f"Best Frobenius coordinates: {frob_coords}")
print(f"\nGround truth: R₀ = β/γ = X1/X2 or equivalently X1*inv(X2)")

## 6. Postprocessing & Pruning

Optimizing rotation and pruning coefficients for the simplest expressions.

In [ ]:
from degeneracy_distillery.postprocessing_utils import postprocess_eqs

X_test = split_data['X_test']
Fs_test = split_data['Fs_test']

pruned_exprs, consts, A_opt = postprocess_eqs(
    coordinates=mdl_coords,
    X=X_test,
    Fs=Fs_test,
    n_params=n_params,
    optimize_rotation="sparse",
    threshold=0.05,
    importance_based=True,
    verbose=True,
)

print("\n" + "="*60)
print("FINAL DISCOVERED EXPRESSIONS")
print("="*60)
for i, expr in enumerate(pruned_exprs):
    print(f"  η_{i+1} = {expr}")
print(f"\nGround truth: One component should approximate β/γ")

## 7. Results & Comparison with Ground Truth

Comparing the discovered symbolic expressions with the known R₀ = β/γ degeneracy.

In [ ]:
# Compare discovered expressions with ground truth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PCA baseline (linear, will fail)
F_avg = Fs_test.mean(0)
eigenvals, eigenvecs = np.linalg.eigh(F_avg)
idx = eigenvals.argsort()[::-1]
eigenvals = eigenvals[idx]
eigenvecs = eigenvecs[:, idx]

ax = axes[0]
ax.scatter(X_test[:, 0], X_test[:, 1], c='gray', alpha=0.1, s=1)
mean_theta = X_test.mean(0)
for i, label in enumerate(['PC1 (well-constrained)', 'PC2 (degenerate)']):
    ev = eigenvecs[:, i]
    scale = 0.3
    ax.arrow(mean_theta[0], mean_theta[1], scale*ev[0], scale*ev[1],
             head_width=0.02, head_length=0.01, fc=f'C{i}', ec=f'C{i}', linewidth=2)
ax.set_xlabel(r'$\beta$')
ax.set_ylabel(r'$\gamma$')
ax.set_title('PCA of Fisher Matrix (Linear)')

# Right: Ground truth R_0 = beta/gamma contours
ax = axes[1]
R0_test = np.array(X_test[:, 0] / X_test[:, 1])
sc = ax.scatter(X_test[:, 0], X_test[:, 1], c=R0_test, s=1, alpha=0.5, cmap='viridis')
plt.colorbar(sc, ax=ax, label=r'$R_0 = \beta/\gamma$')
# Draw constant R_0 lines
for r0 in [1.0, 2.0, 5.0, 10.0]:
    beta_line = np.linspace(BETA_MIN, BETA_MAX, 100)
    gamma_line = beta_line / r0
    mask = (gamma_line >= GAMMA_MIN) & (gamma_line <= GAMMA_MAX)
    ax.plot(beta_line[mask], gamma_line[mask], 'r--', alpha=0.5, linewidth=1)
ax.set_xlabel(r'$\beta$')
ax.set_ylabel(r'$\gamma$')
ax.set_title(r'Ground truth: $R_0 = \beta/\gamma$ contours')

plt.tight_layout()
plt.savefig('sir_results.pdf')
plt.show()